# TOTNet Reliable Colab Inference Notebook (Custom Video)

This notebook is a **reliability-first Colab adapter** for this repository.

It is designed to avoid silent failures:
- verifies paths and checkpoint format
- verifies inference loop actually runs
- verifies overlays are actually drawn
- verifies output video is readable and non-empty
- exports metadata (`.json`) and per-frame predictions (`.csv`)

> Supports input video formats: `.mp4` and `.mov`.

In [ ]:
# 1) Environment setup (Colab)
import os, sys, platform, subprocess
print('Python:', sys.version)
print('Platform:', platform.platform())

# If running in Colab, uncomment installs below as needed.
# %pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# %pip install -q easydict opencv-python numpy pandas tqdm
# !apt-get -y install ffmpeg

In [ ]:
# 2) (Optional) Mount Google Drive
USE_DRIVE = True

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print('Drive mounted at /content/drive')
    except Exception as e:
        print('Drive mount skipped/not available:', e)

In [ ]:
# 3) Repository + path configuration
from pathlib import Path

# If notebook is launched outside repo, set REPO_ROOT accordingly.
REPO_ROOT = Path('/content/TOTNet') if Path('/content/TOTNet').exists() else Path.cwd()
SRC_ROOT = REPO_ROOT / 'src'
WEIGHTS_ROOT = REPO_ROOT / 'weights'

# ---- USER INPUTS ----
INPUT_VIDEO = '/content/input_video.mp4'  # or .mov
OUTPUT_DRIVE_DIR = '/content/drive/MyDrive/TOTNet_outputs' if USE_DRIVE else '/content/TOTNet_outputs'
MODEL_NUM_FRAMES = 5
MODEL_INPUT_HW = (288, 512)  # (H, W) expected by pretrained TOTNet checkpoints in this repo
MODEL_NUM_CHANNELS = 64
WEIGHT_KEYWORD = 'Tennis'  # e.g. 'Tennis', 'Badminton', 'TTA'

assert SRC_ROOT.exists(), f'SRC_ROOT not found: {SRC_ROOT}'
assert WEIGHTS_ROOT.exists(), f'WEIGHTS_ROOT not found: {WEIGHTS_ROOT}'
assert Path(INPUT_VIDEO).suffix.lower() in {'.mp4', '.mov'}, f'Unsupported video format: {INPUT_VIDEO}'
assert Path(INPUT_VIDEO).exists(), f'Input video not found: {INPUT_VIDEO}'

import sys
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print('REPO_ROOT:', REPO_ROOT)
print('INPUT_VIDEO:', INPUT_VIDEO)
print('OUTPUT_DRIVE_DIR:', OUTPUT_DRIVE_DIR)

In [ ]:
# 4) Helpers: robust checkpoint resolve/load and pipeline utilities
import json, csv, time
from collections import deque
from types import SimpleNamespace

import cv2
import numpy as np
import torch

from model.TOTNet import build_motion_model_light

MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(1, 1, 3)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(1, 1, 3)

def resolve_checkpoint(weights_root: Path, keyword: str = '') -> Path:
    cands = sorted(weights_root.rglob('*.pth'))
    assert cands, f'No .pth files under {weights_root}'
    if keyword:
        filtered = [p for p in cands if keyword.lower() in str(p).lower()]
        if filtered:
            # Prefer "best" checkpoint if available
            best = [p for p in filtered if 'best' in p.name.lower()]
            return best[0] if best else filtered[0]
    best = [p for p in cands if 'best' in p.name.lower()]
    return best[0] if best else cands[0]


def strip_module_prefix(state_dict: dict) -> dict:
    out = {}
    for k, v in state_dict.items():
        nk = k[7:] if k.startswith('module.') else k
        out[nk] = v
    return out


def load_totnet_checkpoint(ckpt_path: Path, num_frames: int, num_channels: int, device: torch.device):
    args = SimpleNamespace(num_frames=num_frames, num_channels=num_channels, device=device)
    model = build_motion_model_light(args)
    model.to(device)

    checkpoint = torch.load(str(ckpt_path), map_location=device)
    if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
        raw_state = checkpoint['state_dict']
    elif isinstance(checkpoint, dict):
        raw_state = checkpoint
    else:
        raise RuntimeError(f'Unsupported checkpoint format type: {type(checkpoint)}')

    raw_state = strip_module_prefix(raw_state)
    model_state = model.state_dict()

    matched = {k: v for k, v in raw_state.items() if k in model_state and model_state[k].shape == v.shape}
    missing = [k for k in model_state.keys() if k not in matched]
    unexpected = [k for k in raw_state.keys() if k not in model_state]

    assert len(matched) > 0, 'No checkpoint keys matched model. Wrong checkpoint/model config likely.'

    model_state.update(matched)
    model.load_state_dict(model_state)
    model.eval()

    print(f'Checkpoint loaded: {ckpt_path}')
    print(f'Matched keys: {len(matched)} | Missing model keys: {len(missing)} | Unexpected ckpt keys: {len(unexpected)}')

    return model, {
        'checkpoint': str(ckpt_path),
        'matched_keys': len(matched),
        'missing_model_keys': len(missing),
        'unexpected_ckpt_keys': len(unexpected),
    }


def preprocess_rgb(frame_rgb: np.ndarray, input_hw=(288, 512)) -> np.ndarray:
    h, w = input_hw
    resized = cv2.resize(frame_rgb, (w, h), interpolation=cv2.INTER_LINEAR)
    x = resized.astype(np.float32) / 255.0
    x = (x - MEAN) / STD
    x = np.transpose(x, (2, 0, 1))  # CHW
    return x

In [ ]:
# 5) Run reliable inference: read video -> infer -> draw overlays -> write output + metadata
from datetime import datetime
from pathlib import Path
import pandas as pd

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

ckpt_path = resolve_checkpoint(WEIGHTS_ROOT, WEIGHT_KEYWORD)
model, ckpt_stats = load_totnet_checkpoint(ckpt_path, MODEL_NUM_FRAMES, MODEL_NUM_CHANNELS, device)

run_id = datetime.utcnow().strftime('totnet_%Y%m%d_%H%M%S')
out_root = Path('/tmp') / run_id
out_root.mkdir(parents=True, exist_ok=True)
frames_dir = out_root / 'preview_frames'
frames_dir.mkdir(exist_ok=True)

video_in = Path(INPUT_VIDEO)
cap = cv2.VideoCapture(str(video_in))
assert cap.isOpened(), f'Failed to open input video: {video_in}'

input_fps = cap.get(cv2.CAP_PROP_FPS)
input_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
input_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
input_n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f'Input video stats: {input_w}x{input_h} @ {input_fps:.3f} fps, frames={input_n}')
assert input_w > 0 and input_h > 0, 'Invalid input video dimensions.'
assert input_n > MODEL_NUM_FRAMES, f'Video too short for num_frames={MODEL_NUM_FRAMES}'

out_video_tmp = out_root / f'{video_in.stem}_annotated.mp4'
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(str(out_video_tmp), fourcc, input_fps if input_fps > 1 else 30.0, (input_w, input_h))
assert writer.isOpened(), 'Failed to initialize VideoWriter.'

seq = deque(maxlen=MODEL_NUM_FRAMES)
records = []

frames_read = 0
frames_inferred = 0
overlays_drawn = 0
pixels_changed_total = 0
forward_time_sum = 0.0

while True:
    ret, frame_bgr = cap.read()
    if not ret:
        break

    frames_read += 1
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    seq.append(preprocess_rgb(frame_rgb, MODEL_INPUT_HW))

    annotated = frame_bgr.copy()
    pred_x_full, pred_y_full, conf = None, None, None

    if len(seq) == MODEL_NUM_FRAMES:
        x = np.stack(seq, axis=0)  # [N,C,H,W]
        x = torch.from_numpy(x).unsqueeze(0).to(device=device, dtype=torch.float32)  # [1,N,C,H,W]

        t0 = time.perf_counter()
        with torch.no_grad():
            out = model(x)
        torch.cuda.synchronize() if device.type == 'cuda' else None
        forward_time_sum += (time.perf_counter() - t0)
        frames_inferred += 1

        # TOTNet forward returns [B, H*W] probabilities in this repo.
        if out.ndim == 2:
            heat = out[0]
            flat_idx = int(torch.argmax(heat).item())
            model_h, model_w = MODEL_INPUT_HW
            pred_x = flat_idx % model_w
            pred_y = flat_idx // model_w
            conf = float(heat[flat_idx].item())
        elif out.ndim == 3:
            heat2d = out[0]
            pred_y, pred_x = np.unravel_index(int(torch.argmax(heat2d).item()), tuple(heat2d.shape))
            conf = float(torch.max(heat2d).item())
        else:
            raise RuntimeError(f'Unexpected model output shape: {tuple(out.shape)}')

        pred_x_full = int(round(pred_x * input_w / MODEL_INPUT_HW[1]))
        pred_y_full = int(round(pred_y * input_h / MODEL_INPUT_HW[0]))
        pred_x_full = int(np.clip(pred_x_full, 0, input_w - 1))
        pred_y_full = int(np.clip(pred_y_full, 0, input_h - 1))

        before = annotated.copy()
        cv2.circle(annotated, (pred_x_full, pred_y_full), 9, (255, 0, 255), -1)
        cv2.putText(
            annotated,
            f'TOTNet conf={conf:.4f}',
            (20, 36),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.9,
            (0, 255, 0),
            2,
            cv2.LINE_AA,
        )
        changed = int(np.count_nonzero(cv2.absdiff(before, annotated)))
        if changed > 0:
            overlays_drawn += 1
            pixels_changed_total += changed

    writer.write(annotated)

    records.append({
        'frame_index': frames_read - 1,
        'inference_executed': len(seq) == MODEL_NUM_FRAMES,
        'pred_x': pred_x_full,
        'pred_y': pred_y_full,
        'confidence': conf,
    })

    if frames_read <= 12:
        cv2.imwrite(str(frames_dir / f'frame_{frames_read:04d}.jpg'), annotated)

cap.release()
writer.release()

# --- Hard failure checks (silent-failure protection) ---
assert frames_read > 0, 'No frames read from input video.'
assert frames_inferred > 0, 'Inference was never executed (empty inference loop).' 
assert overlays_drawn > 0, 'No visible overlays were drawn.'
assert out_video_tmp.exists() and out_video_tmp.stat().st_size > 10_000, 'Output video missing or too small.'

# Re-open output to verify readable file and frame count
verify = cv2.VideoCapture(str(out_video_tmp))
assert verify.isOpened(), 'Output video cannot be reopened for verification.'
out_n = int(verify.get(cv2.CAP_PROP_FRAME_COUNT))
out_w = int(verify.get(cv2.CAP_PROP_FRAME_WIDTH))
out_h = int(verify.get(cv2.CAP_PROP_FRAME_HEIGHT))
verify.release()
assert out_n > 0 and out_w == input_w and out_h == input_h, 'Output video verification failed.'

# Save metadata locally
pred_csv = out_root / f'{video_in.stem}_predictions.csv'
meta_json = out_root / f'{video_in.stem}_metadata.json'

pd.DataFrame(records).to_csv(pred_csv, index=False)

meta = {
    'run_id': run_id,
    'input_video': str(video_in),
    'input_video_stats': {'width': input_w, 'height': input_h, 'fps': input_fps, 'frames': input_n},
    'model_input_hw': MODEL_INPUT_HW,
    'model_num_frames': MODEL_NUM_FRAMES,
    'checkpoint_stats': ckpt_stats,
    'frames_read': frames_read,
    'frames_inferred': frames_inferred,
    'overlays_drawn': overlays_drawn,
    'pixels_changed_total': pixels_changed_total,
    'avg_forward_ms': (forward_time_sum / max(frames_inferred, 1)) * 1000,
    'output_video_tmp': str(out_video_tmp),
    'output_video_verified': {'frames': out_n, 'width': out_w, 'height': out_h},
    'predictions_csv': str(pred_csv),
}
with open(meta_json, 'w') as f:
    json.dump(meta, f, indent=2)

# Copy final artifacts to Drive/output directory
final_dir = Path(OUTPUT_DRIVE_DIR)
final_dir.mkdir(parents=True, exist_ok=True)
final_video = final_dir / out_video_tmp.name
final_csv = final_dir / pred_csv.name
final_json = final_dir / meta_json.name

import shutil
shutil.copy2(out_video_tmp, final_video)
shutil.copy2(pred_csv, final_csv)
shutil.copy2(meta_json, final_json)

print('=== SUCCESS ===')
print('Final video:', final_video)
print('Predictions CSV:', final_csv)
print('Metadata JSON:', final_json)
print('Frames read / inferred / overlays drawn:', frames_read, frames_inferred, overlays_drawn)
print('Avg forward time (ms):', round(meta['avg_forward_ms'], 3))

In [ ]:
# 6) Quick preview (works in Colab/Jupyter)
from IPython.display import Video, display
display(Video(str(final_video), embed=True, width=960))

## Reliability Checklist (what this notebook explicitly verifies)

1. Input video path exists and extension is `.mp4`/`.mov`.
2. Weights folder has at least one checkpoint and key matching is non-zero.
3. Video can be decoded; dimensions and frame count are valid.
4. Inference loop actually executes (`frames_inferred > 0`).
5. Overlay drawing changes pixels (`overlays_drawn > 0`).
6. Output video is written, non-empty, and can be decoded again.
7. Prediction CSV + metadata JSON are exported.
8. Final artifacts are copied to Google Drive output directory.